# Challenge 5: Classifying Particles from Detector Signatures

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Preprocessing](#44-preprocessing)
5. [The Challenge](#5-the-challenge)
   - [5.1 Training the Model](#51-training-the-model)
   - [5.2 Evaluating the Model](#52-evaluating-the-model)
   - [5.3 Interpretation](#53-interpretation)
   - [5.4 Reflection Questions](#54-reflection-questions)
6. [Solution](#6-solution)
7. [References](#7-references)

---

## 1. Introduction

Every time two proton beams collide inside a detector such as ATLAS or CMS at
CERN's Large Hadron Collider (LHC), several hundred particles can spray outward
from the collision point in a fraction of a nanosecond. These particles are not
observed directly. Instead, the detector records the signatures they leave behind:
tracks bent by magnetic fields, energy deposits (or *clusters*) in calorimeters,
and hits in specialised detector systems such as muon chambers. From these raw
signals, sophisticated reconstruction algorithms attempt to determine which
particles were produced and what their properties were, including quantities such
as momentum, energy, and electric charge.

Classifying those particles correctly is an important problem in experimental
particle physics. If you misidentify a pion as an electron, for example, you
introduce background contamination into measurements involving electrons, such
as determinations of the W boson production cross-section. If you fail to
identify muons cleanly, you may lose some of the most useful signatures for
discovering rare processes or searching for new particles beyond the Standard
Model.

This challenge focuses on distinguishing three particle types that appear
routinely in general-purpose detectors:

- **Electrons** leave a narrow, high-energy shower almost entirely in the
  electromagnetic calorimeter, the detector subsystem designed to absorb and
  measure electrons and photons. They are relatively rare in inclusive
  proton-proton collisions but appear prominently in decays involving W and Z
  bosons.
  
- **Muons** pass through nearly all detector material without producing large
  particle showers. They leave a clean, long track and deposit only a small
  amount of energy through ionisation (the process of removing electrons from
  atoms as they pass through matter). Because they can penetrate large amounts
  of material, muons are often detected in specialised chambers located at the
  outer edge of the detector. They are the majority class here because inclusive
  QCD (Quantum Chromodynamics) processes produce them copiously.

- **Pions** are the most common charged hadrons (particles composed of quarks
  and bound together by the strong nuclear force). Unlike electrons, they
  interact strongly with detector material, depositing much of their energy in
  the hadronic calorimeter. At low momenta, some of their signatures overlap
  with those of electrons, making the electron/pion boundary the most difficult
  classification problem in this dataset.

The problem is non-trivial for several reasons. The feature distributions
overlap: at low momentum, electrons and pions can have similar track curvatures
and similar total calorimeter energy deposits. Several features are also
physically correlated with one another. In particular, the ratio of
electromagnetic to hadronic energy is calculated directly from the two
calorimeter energy measurements and is therefore mathematically related to both.
This creates a degree of collinearity that can affect the behaviour of some
machine learning algorithms.

Another important feature is the ionisation energy loss per unit length,
commonly written as **dE/dx**. This quantity is governed by the **Bethe-Bloch
equation**, which describes how charged particles lose energy as they pass
through matter. At the relativistic energies that dominate this sample, dE/dx
tends to be inversely correlated with momentum, introducing another physically
meaningful relationship into the dataset.

These correlations are not artefacts or mistakes; they arise from the underlying
physics of particle interactions and detector response. However, they do affect
how linear classifiers behave, which is something you will need to account for
when building and evaluating your models.

### Learning Objectives

- Apply multinomial logistic regression (softmax regression) to a
  three-class particle classification problem
- Recognise when and why feature scaling is required for logistic regression
  with gradient-based optimisation, and apply it correctly
- Evaluate a multiclass classifier using confusion matrices, per-class
  precision, recall, and F1 scores rather than accuracy alone
- Interpret the learned coefficient matrix of a logistic regression model
  in the context of the physics features
- Handle physically correlated features and reason about what effect they
  have on coefficient estimates

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `LogisticRegression` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html |
| Multiclass strategies in scikit-learn | https://scikit-learn.org/stable/modules/multiclass.html |
| `StandardScaler` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html |
| `confusion_matrix` and `ConfusionMatrixDisplay` | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html |
| `classification_report` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| Particle Data Group Review of Particle Physics | https://pdg.lbl.gov |
| ATLAS detector overview (CERN) | https://atlas.cern/discover/detector |
| The Bethe-Bloch formula and dE/dx (PDG) | https://pdg.lbl.gov/2023/reviews/rpp2023-rev-passage-particles-matter.pdf |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook was written for Python 3.9 or later. All required libraries
are part of the standard scientific Python stack and are available in any
Anaconda or pip-based environment that includes scikit-learn. No GPU or
specialist physics software is required.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations and array handling |
| `pandas` | Loading and inspecting the dataset |
| `matplotlib` | Plotting and visualisation |
| `seaborn` | Statistical plots, including heatmaps and distribution plots |
| `sklearn.model_selection` | Train/test splitting with stratification |
| `sklearn.preprocessing` | Feature scaling via `StandardScaler` |
| `sklearn.linear_model` | `LogisticRegression` classifier |
| `sklearn.metrics` | Confusion matrix, classification report, accuracy score |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `particles.csv` |
| Location | `data/particles.csv` (relative to this notebook) |
| Total rows | 10,000 |
| Features | 8 continuous/quasi-continuous columns |
| Target column | `particle_type` |
| Class 0 | electron: 2,043 samples (~20%) |
| Class 1 | muon: 4,940 samples (~50%) |
| Class 2 | pion: 3,017 samples (~30%) |

The class distribution is moderately imbalanced: the majority class (muon)
is 2.5x more frequent than the minority class (electron). This is realistic.
In typical LHC data, muon rates from inclusive QCD production far exceed
the electron rate in the same kinematic range.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
)

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/particles.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(10)

### 4.2 Understanding the Features

The eight features are loosely inspired by measurements available in a
general-purpose particle detector. They are not raw detector readouts;
they represent reconstructed quantities derived from lower-level signals.

| Feature | Type | Units | Description |
|---|---|---|---|
| `momentum_gev` | Continuous | GeV/c | Reconstructed track momentum. Determined from the curvature of the track in the magnetic field. Muons in inclusive production span a wide range; electrons and pions are softer on average. |
| `track_chi2` | Continuous | dimensionless | Chi-squared per degree of freedom of the track fit. A value near 1 indicates a good fit. Hadronic interactions can scatter pions in the tracker material, producing worse fits (higher values). |
| `ecal_energy` | Continuous | GeV | Energy deposited in the electromagnetic calorimeter. Electrons shower almost entirely here. Pions deposit moderate amounts via pi-zero decay to photons. Muons deposit very little (minimum ionising). |
| `hcal_energy` | Continuous | GeV | Energy deposited in the hadronic calorimeter. Pions shower strongly here. Muons deposit little. Electrons deposit near zero (they stop in the ECAL). |
| `ecal_hcal_ratio` | Continuous | dimensionless | Ratio of ECAL energy to total calorimeter energy: ecal / (ecal + hcal). Near 1 for electrons, near 0 for pions, intermediate for muons (both deposits are small). This feature is derived from `ecal_energy` and `hcal_energy` and is therefore correlated with both. |
| `dE_dx` | Continuous | MeV/cm | Ionisation energy loss per unit length in the tracker. At relativistic momenta this follows the Bethe-Bloch relation and decreases with increasing momentum before rising again. It is correlated with `momentum_gev` and `track_chi2`. |
| `isolation_score` | Continuous | dimensionless | Energy in a cone of fixed angular radius around the particle, divided by the particle's energy. Low values indicate an isolated particle (typical for leptons from W/Z decays). High values indicate the particle sits inside a jet (more typical for pions). |
| `n_hits` | Quasi-continuous | count | Number of tracker hits associated with the track. Muons traverse the full detector and accumulate the most hits. Pions may interact and stop early, reducing the count. |
| `particle_type` | Categorical | label | Target variable. 0 = electron, 1 = muon, 2 = pion. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
# Class counts and proportions
print('Class distribution:')
counts = df['particle_type'].value_counts().sort_index()
names = {0: 'electron', 1: 'muon', 2: 'pion'}
for label, count in counts.items():
    print(f'  Class {label} ({names[label]:10s}): {count:5d}  ({100*count/len(df):.1f}%)')

print()
print('Summary statistics:')
df.describe().round(3)

**Questions to consider after examining the output above:**

- What is the ratio of the majority class (muon) to the minority class
  (electron)? Is this a severe imbalance or a moderate one?
- Look at the standard deviations relative to the means. Which features
  have the widest spread? Does that make physical sense given the
  descriptions above?
- Accuracy calculated on the full dataset would heavily reflect performance
  on the muon class. What evaluation strategy is better suited to this
  three-class setting?

In [ ]:
# Listing 4.
# Feature distributions by class
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
features = [c for c in df.columns if c != 'particle_type']
colours = {0: 'steelblue', 1: 'darkorange', 2: 'seagreen'}
labels_map = {0: 'electron', 1: 'muon', 2: 'pion'}

for ax, feat in zip(axes, features):
    for cls in [0, 1, 2]:
        subset = df.loc[df['particle_type'] == cls, feat]
        subset.plot(kind='kde', ax=ax, label=labels_map[cls],
                    color=colours[cls], linewidth=1.8)
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Particle Type', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- Which features show the clearest separation between all three classes?
  Which features have the most overlap?
- `ecal_energy` and `ecal_hcal_ratio` both appear to separate electrons
  from pions. Why would they behave similarly, and what consequence might
  this have for a linear model?
- The electron and pion distributions overlap substantially in `momentum_gev`.
  Based on the physics description, which other features do you expect the
  model to rely on to resolve this ambiguity?

In [ ]:
# Listing 5.
# Correlation matrix of all features
corr = df.drop(columns='particle_type').corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True
)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

**Questions to consider:**

- Identify the two most strongly correlated feature pairs. Can you explain
  the correlation physically, using the descriptions in Section 4.2?
- `ecal_hcal_ratio` is derived from `ecal_energy` and `hcal_energy`. What
  does strong multicollinearity between these three features mean for the
  magnitude of the logistic regression coefficients?
- `dE_dx` is correlated with `momentum_gev`. At high momenta, particles
  become relativistic and their ionisation loss decreases. Does the sign of
  the correlation in the matrix match this expectation?

### 4.4 Preprocessing

As covered in Notebook_5, logistic regression with the default `lbfgs` or
`saga` solver optimises a loss function using gradient-based methods. These
solvers are sensitive to the scale of the input features: when one feature
spans a range of 0 to 160 (momentum in GeV/c) and another spans 0 to 1
(isolation score), the gradient updates for the two features are on entirely
different scales. This slows convergence, sometimes to the point that the
solver fails to reach a good solution within the default iteration limit.
Standardising the features (zero mean, unit variance) puts all inputs on
the same scale and fixes this.

The critical rule: fit the scaler on the training data only, then transform
both training and test sets using the parameters learned from training. If
you fit the scaler on the whole dataset before splitting, you allow test-set
statistics to influence the training process. This is a data leakage error
that inflates reported performance.

The class distribution here (20/50/30) is moderately imbalanced but not
extreme. Stratified splitting preserves the class ratios in both the
training and test sets, which matters when the minority class has only
2,000 samples.

In [ ]:
# Listing 6.
# TODO: Implement preprocessing.
#
# 1. Separate features (X) from the target column ('particle_type') (y).
#
# 2. Split X and y into training and test sets.
#    Use an 80/20 split and set random_state=42 for reproducibility.
#    Think carefully: should you use stratification here? Why?
#
# 3. Apply StandardScaler to the features.
#    Remember: fit only on X_train, then transform both X_train and X_test.
#    Do not fit on the whole dataset or on X_test.
#
# 4. Print the shapes of X_train, X_test, y_train, y_test after splitting
#    to confirm the split looks correct.
#    Also print the class distribution in y_train and y_test to confirm
#    stratification worked as expected.

# YOUR CODE HERE

---
## 5. The Challenge

Your task is to train a multinomial logistic regression classifier to
distinguish electrons, muons, and pions using the eight detector features.

`LogisticRegression` in scikit-learn handles multiclass problems
automatically. With the `lbfgs` solver and more than two classes,
the model uses a softmax formulation and estimates a separate set of
coefficients for each class, outputting a probability distribution
over the three classes for each input. This is exactly the setting
described in Notebook_5.

Because the features have different physical units and dynamic ranges,
the scaled data from Section 4.4 must be used here. The solver will not
converge reliably on the raw features.

### 5.1 Training the Model

In [ ]:
# Listing 7.
# TODO: Train the logistic regression model.
#
# 1. Instantiate LogisticRegression with appropriate parameters.
#    Key choices to make:
#      - solver: 'lbfgs' handles multiclass problems natively and works
#        well with moderate-sized datasets. Alternatively, 'saga' supports
#        additional regularisation options.
#      - C: the inverse regularisation strength. What does a larger C mean
#        for the bias-variance trade-off? Start with C=1.0.
#      - max_iter: the default (100) may not be enough. Try 1000.
#      - random_state: set to 42 for reproducibility.
#    Note: in scikit-learn 1.x, logistic regression automatically uses
#    a softmax (multinomial) formulation when there are more than two
#    classes with the 'lbfgs' solver. No extra parameter is needed.
#
# 2. Fit the model to the scaled training data (X_train_scaled, y_train).
#
# 3. After fitting, print:
#      - The number of iterations the solver took to converge
#        (check the model's .n_iter_ attribute)
#      - The shape of the coefficient matrix (.coef_)
#      - What does the shape of .coef_ tell you about how the model handles
#        three classes?

# YOUR CODE HERE

### 5.2 Evaluating the Model

For a three-class problem, overall accuracy gives you a single number that
conflates performance across all classes. If the model correctly classifies
97% of muons but only 60% of electrons, accuracy might still look
respectable because muons are the majority class. Per-class precision,
recall, and F1 score tell you where the model is succeeding and where it
is failing.

- **Precision** for a class: of all predictions made for that class, what
  fraction were correct? High precision means few false positives.
- **Recall** for a class: of all true examples of that class, what fraction
  were found? High recall means few false negatives.
- **F1 score**: the harmonic mean of precision and recall, useful when you
  want a single per-class summary.

The confusion matrix shows the full picture: how often each true class is
predicted as each other class. Off-diagonal entries are misclassifications.
For particle physics, the electron row of the confusion matrix is
particularly important: electrons misidentified as pions would corrupt
electroweak measurements.

In [ ]:
# Listing 8.
# TODO: Evaluate the model.
#
# 1. Generate predictions on the scaled test set.
#
# 2. Compute and print the overall accuracy score.
#
# 3. Print the full classification report using classification_report().
#    Pass target_names=['electron', 'muon', 'pion'] so the output is readable.
#    Examine precision, recall, and F1 for each class separately.
#
# 4. Plot the confusion matrix.
#    Use ConfusionMatrixDisplay.from_predictions() or equivalent.
#    Pass display_labels=['electron', 'muon', 'pion'].
#    Which class is most often confused with which other class?

# YOUR CODE HERE

### 5.3 Interpretation

A logistic regression model with `multi_class='multinomial'` learns one
coefficient vector per class. The coefficient matrix has shape
(n_classes, n_features), so for this problem it is (3, 8). Each row
contains the weights assigned to each feature when predicting that class
against the softmax baseline.

After scaling, coefficients are directly comparable across features:
a large positive coefficient for class 1 (muon) on feature `ecal_energy`
means that a high ECAL deposit pushes the model toward predicting muon.
If that contradicts the physics (muons deposit very little in the ECAL),
it likely reflects the collinearity structure of the features pulling
the coefficient in an unexpected direction.

Inspecting the coefficient matrix is therefore both a model diagnostic
and a physics check.

In [ ]:
# Listing 9.
# TODO: Interpret the model coefficients.
#
# 1. Extract the coefficient matrix from the fitted model (.coef_).
#    It has shape (3, 8): one row per class, one column per feature.
#
# 2. Create a pandas DataFrame from the coefficient matrix.
#    Use the feature names as column names and ['electron', 'muon', 'pion']
#    as the row index.
#    Print this DataFrame.
#
# 3. Create a heatmap of the coefficient matrix using seaborn.heatmap().
#    Use a diverging colour map (e.g. 'coolwarm') so that positive and
#    negative coefficients are visually distinct.
#    Label the axes clearly.
#
# 4. Look at the electron row. Which features have the largest positive
#    coefficients for the electron class? Do they match what you would
#    expect from the physics description in Section 4.2?
#
# 5. Look at the muon row. Which features have large negative coefficients?
#    What does a large negative coefficient mean: the model treats a high
#    value of that feature as evidence *against* muon?

# YOUR CODE HERE

### 5.4 Reflection Questions

Take a few minutes to think through these before moving to the solution.
They are not assessed but will help you understand what the model is and
is not doing.

1. The `ecal_hcal_ratio` feature is mathematically derived from
   `ecal_energy` and `hcal_energy`. If you removed it from the feature
   set, do you expect model performance to change significantly? Why or
   why not? How would you test this empirically?

2. Logistic regression assumes a linear decision boundary in feature space.
   Given the KDE plots in Section 4.3, do you think linear boundaries are
   adequate here, or do the class distributions suggest a non-linear
   classifier might do better?

3. The model is trained with the default regularisation strength (C=1.0).
   Given the correlated features in this dataset, what effect might stronger
   regularisation (smaller C) have on the coefficient matrix? Would it
   improve or worsen test performance?

4. Suppose this model is deployed in a real trigger system that decides
   in real time which detector events to keep for further analysis.
   Which type of error is more costly: classifying a pion as an electron,
   or classifying an electron as a pion? How would you adjust the model
   to reflect this asymmetry?

5. The dataset contains ~2% label noise. How does this affect the
   theoretical upper bound on classification accuracy? Is the performance
   you observed consistent with a model that is near that upper bound?

---
## 6. Solution

**Read this section only after attempting the challenge yourself.**
The solution works through every step with explanations of the reasoning
behind each decision. Run the cells from top to bottom; they are
self-contained and do not depend on any variables you may have defined
in Section 5.

### Step 1: Preprocessing

We separate features from the target, split the data, and scale.

**Why 80/20?** With 10,000 samples and three classes, an 80/20 split gives
2,000 test samples, which is enough for stable per-class metrics. The
minority class (electron) contributes around 400 test samples at this split,
which is sufficient for meaningful recall and precision estimates.

**Why stratification?** Without it, random chance could place a
disproportionate share of the 2,000 electrons in one split. Stratification
guarantees the class ratios are preserved. For balanced datasets this makes
little difference, but for the minority class here it matters.

**Why scale?** `lbfgs` uses gradient information to update the coefficients.
Raw `momentum_gev` values range up to 160 GeV/c while `isolation_score`
ranges from 0 to 1. The gradient with respect to the momentum feature is
therefore much smaller than the gradient for isolation, causing the solver
to take very small steps for momentum and very large steps for isolation.
Standardisation equalises the scale and allows the solver to converge
efficiently.

In [ ]:
# Listing 10.
feature_cols = [c for c in df.columns if c != 'particle_type']
X = df[feature_cols].values
y = df['particle_type'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set:  X={X_train_sc.shape}, y={y_train.shape}')
print(f'Test set:      X={X_test_sc.shape},  y={y_test.shape}')

print('\nClass distribution in training set:')
for cls in [0, 1, 2]:
    n = np.sum(y_train == cls)
    print(f'  {names[cls]:10s}: {n} ({100*n/len(y_train):.1f}%)')

print('\nClass distribution in test set:')
for cls in [0, 1, 2]:
    n = np.sum(y_test == cls)
    print(f'  {names[cls]:10s}: {n} ({100*n/len(y_test):.1f}%)')

The class ratios are preserved in both splits, confirming stratification
worked as intended.

### Step 2: Training the Model

We use `lbfgs`, which in scikit-learn 1.x automatically applies a softmax
(multinomial) formulation when there are more than two classes. This fits
a single set of coefficients per class using the softmax function, and
outputs a probability distribution over the three classes for each input.
This is more principled than fitting three independent binary classifiers
(one-vs-rest) when the classes are genuinely mutually exclusive, as they
are here: a particle can only be one type.

`C=1.0` is the scikit-learn default, applying L2 regularisation with
moderate strength. Given that we have correlated features, regularisation
is helpful: it prevents individual coefficients from growing large to
compensate for collinearity.

`max_iter=1000` is a generous upper bound; in practice the solver will
converge well before this on the scaled data.

In [ ]:
# Listing 11.
clf = LogisticRegression(
    solver='lbfgs',
    C=1.0,
    max_iter=1000,
    random_state=42
)
clf.fit(X_train_sc, y_train)

print(f'Solver converged in {clf.n_iter_[0]} iterations.')
print(f'Coefficient matrix shape: {clf.coef_.shape}')
print('Shape is (n_classes, n_features) = (3, 8).')
print('Each row holds the coefficients for one class in the softmax.')

### Step 3: Evaluating the Model

Overall accuracy is reported first for context, but the classification
report is what matters. With three classes and a 20/50/30 split, a
classifier that predicted "muon" for everything would achieve 50% accuracy.
Macro-averaged F1 (the unweighted mean of per-class F1 scores) is a more
honest summary: it gives equal weight to all three classes regardless of
their frequency.

In [ ]:
# Listing 12.
y_pred = clf.predict(X_test_sc)

acc = accuracy_score(y_test, y_pred)
print(f'Overall accuracy: {acc:.4f}\n')
print('Classification report:')
print(classification_report(y_test, y_pred,
                            target_names=['electron', 'muon', 'pion']))

**Interpreting the results:**

Muons are classified most accurately, as expected: their combination of
low calorimeter deposits and high hit counts is distinctive. Electrons
and pions are harder to separate; the model tends to confuse them at
low momentum where their signatures overlap. The electron recall is
typically the weakest metric, meaning some electrons are lost to
misclassification as pions. In a real physics analysis, this
misidentification rate would be quoted as the "fake rate" and would
need to be controlled by tightening identification cuts, at the cost
of electron efficiency (recall).

In [ ]:
# Listing 13.
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['electron', 'muon', 'pion'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix: Particle Classification\n(LogisticRegression, multinomial)', fontsize=11)
plt.tight_layout()
plt.show()

The confusion matrix shows which class pairs are most problematic.
The dominant off-diagonal entries are almost always in the
electron/pion pair, confirming that the electron-pion boundary is
the hardest to draw. Muon rows and columns are nearly clean.

### Step 4: Interpreting the Coefficient Matrix

The coefficient matrix has shape (3, 8). Row `i` contains the weights
assigned to each feature when computing the log-odds score for class `i`.
After scaling, all features are on the same scale, so coefficient
magnitudes are directly comparable within a row.

A large positive coefficient for class 0 (electron) on `ecal_hcal_ratio`
makes physical sense: electrons shower primarily in the ECAL, giving a
high ratio. A large negative coefficient for class 2 (pion) on the same
feature also makes sense: pions deposit more in the HCAL, giving a
low ratio.

Where you may see physically unexpected signs, look to collinearity.
`ecal_hcal_ratio` is derived from `ecal_energy` and `hcal_energy`.
When all three are in the model together, the solver distributes
predictive weight across the correlated triplet in ways that can
produce coefficients with unintuitive signs, even though the overall
predictions remain correct.

In [ ]:
# Listing 14.
coef_df = pd.DataFrame(
    clf.coef_,
    index=['electron', 'muon', 'pion'],
    columns=feature_cols
)
print('Coefficient matrix (one row per class, one column per feature):')
print(coef_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(
    coef_df, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Logistic Regression Coefficients by Class and Feature', fontsize=12)
ax.set_xlabel('Feature')
ax.set_ylabel('Particle class')
plt.tight_layout()
plt.show()

**Key observations from the coefficient heatmap:**

- `ecal_hcal_ratio` has a large positive electron coefficient and a large
  negative pion coefficient, consistent with the physics.
- `isolation_score` has a large negative electron coefficient (electrons
  are isolated, low score = evidence for electron) and a positive pion
  coefficient (pions sit inside jets, high score = evidence for pion).
- `hcal_energy` has the largest positive pion coefficient, consistent with
  pions showering hadronically.
- Muon coefficients are generally moderate in magnitude: muons are
  identified partly by the *absence* of calorimeter activity, which
  shows up as negative coefficients on the calorimeter features combined
  with a positive `n_hits` coefficient.
- Some coefficients may appear to contradict simple physics intuition.
  Before concluding that the model is wrong, check whether the feature
  in question is strongly correlated with another feature in the model.
  Collinear features share predictive weight in ways that individual
  coefficients do not cleanly reflect.

---
## 7. References

1. Cox, D.R. (1958). The regression analysis of binary responses.
   *Journal of the Royal Statistical Society, Series B*, 20(2), 215-242.
   The foundational paper for logistic regression.

2. Particle Data Group. (2023). *Review of Particle Physics*.
   *Progress of Theoretical and Experimental Physics*, 2022(8), 083C01.
   https://pdg.lbl.gov
   Authoritative reference for particle properties and detector physics.

3. ATLAS Collaboration. (2008). The ATLAS Experiment at the CERN Large
   Hadron Collider. *Journal of Instrumentation*, 3, S08003.
   https://doi.org/10.1088/1748-0221/3/08/S08003
   Reference for the detector context motivating this classification problem.

4. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html
   Cite this when reporting results obtained with scikit-learn.

5. Hastie, T., Tibshirani, R., and Friedman, J. (2009).
   *The Elements of Statistical Learning* (2nd ed.), Chapter 4.
   Springer. https://web.stanford.edu/~hastie/ElemStatLearn/
   Covers multinomial logistic regression and its properties in depth.